# Multi-Agent ulcer Diet Recommendation System with RAG.

#### Architecture

User -> Food Safety Agent -> Food Knowledge RAG (Vector DB) -> Nigerian Dish Agent -> Nutrition Reasoning Agent -> Final Recommendation Agent -> UI (Gradio)

#### Technologies:

- LLM access → OpenRouter
- Vector database → ChromaDB
- UI → Gradio
- Embeddings → sentence-transformers

#### A small Nigerian Food Knowledge Dataset

- Create a file containing your data, and name it `ulcer_foods.txt`
- Add it to this same folder

Example of data: check link: https://drive.google.com/file/d/1GHV05EFTjG75v6IuWO60gGOQnk1az-od/view?usp=sharing

In [2]:
# Create Vector Database (RAG)

import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="diet_db")
collection = client.get_or_create_collection("ulcer_foods")

with open("ulcer_foods.txt") as f:
    foods = f.readlines()

for i, food in enumerate(foods):
    embedding = embed_model.encode(food).tolist()

    collection.add(
        documents=[food],
        embeddings=[embedding],
        ids=[str(i)]
    )

In [3]:
# Retrieval Function

def retrieve_foods(query):

    embedding = embed_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[embedding],
        n_results=5
    )

    return "\n".join(results["documents"][0])

In [4]:
# OpenAI Client

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [10]:
# Food Safety Agent

def food_safety_agent(query):

    context = retrieve_foods(query)

    prompt = f"""
You are a medical food safety assistant.

User condition: stomach ulcer.

Food knowledge:
{context}

User question:
{query}

Return:
- foods safe
- foods to avoid
"""

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return r.choices[0].message.content

In [16]:
# Nigerian Dish Agent

def nigerian_food_agent(safety):

    prompt = f"""
Based on this safety advice:

{safety}

Suggest Nigerian dishes for:

Breakfast
Lunch
Dinner
"""

    r = client.chat.completions.create(
        model="meta-llama/llama-3.1-70b-instruct",
        messages=[{"role":"user","content":prompt}]
    )

    return r.choices[0].message.content

In [15]:
# Nutrition Reasoning Agent

def nutrition_reasoning_agent(meals):

    prompt = f"""
Explain why these meals are safe for ulcer patients.

{meals}
"""

    r = client.chat.completions.create(
        model="deepseek/deepseek-v3.2",
        messages=[{"role":"user","content":prompt}]
    )

    return r.choices[0].message.content

In [17]:
# Diet Recommendation Agent

def diet_recommendation(query):

    safety = food_safety_agent(query)

    meals = nigerian_food_agent(safety)

    reasoning = nutrition_reasoning_agent(meals)

    return f"""
ULCER DIET RECOMMENDATION

{safety}

Recommended Nigerian Meals
{meals}

Why These Foods Work
{reasoning}
"""

In [19]:
# UI (Gradio)

import gradio as gr

def run(query):
    return diet_recommendation(query)

ui = gr.Interface(
    fn=run,
    inputs=gr.Textbox(label="Ask about ulcer diet"),
    outputs=gr.Markdown(),
    title="Ulcer Diet AI Assistant"
)

ui.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
